In [ ]:
include( "../model.jl" )
include( "../prob.jl" )

using ThreadsX
using StatsBase
using Printf

using StatsPlots
using Distributions

home, base = expfolder( "system" )

function average(N::defInt, x::Vector{defFloat})::defFloat
    return sum( (1:N).*x )
end

function abundance(x::Vector{defFloat}; a::defInt=2)::defFloat
    ι = 0.0 .< x
    return a == 1 ? exp( -sum( x[ι].*log.( x[ι] ) ) ) : sum( x[ι].^a )^(1/(1 - a))
end

function evenness(x::Vector{defFloat}; a::defInt=2, b::defInt=1)::defFloat
    return abundance( x; a=a )/abundance( x; a=b )
end;

In [ ]:
# System parameters.
N = 1024
ρ = 0.251

# Variable effective temperature.
δη = 0.025;  ηlist = [0.01;δη:δη:0.75]
ηmin = minimum( ηlist );  ηmax = maximum( ηlist )
Nη = length( ηlist )

# Variable system density.
δs = 0.1;  smin = -2.0;  smax = 0.0
slist = 10.0.^(smin:δs:smax)
Ns = length( slist )

In [ ]:
# Initialize folder names.
folderdata = [findfolder( N, ρ, slist[i], ηlist[j]; base=base ) for i ∈ 1:Ns, j ∈ 1:Nη]

# Import order parameter data.
T = 10_000
φdatalist = vec.( readdlm.( folderdata.*"order_$(T)-end.txt" ) )

# Order statistics.
μφdata = mean.( φdatalist )
σφdata = std.(  φdatalist )

# Compute the critical noise.
ηclist = [criticalnoise( ηlist, μφdata[i,:] ) for i ∈ 1:Ns];

In [ ]:
# Build heatmap of the optimal JSD.
plt = plot(; size=(350,300) )
plot!( plt; right_margin=15pt, bottom_margin=0pt )

heatmap!( plt, slist, ηlist, μφdata'; cmap=colormap1, clims=(0,1) )
plot!( plt, slist, ηclist; color=:gray70, lw=1, marker=:circ, label="" )

plot!( plt; xlims=10.0.^(-2,0), xscale=:log10 )
plot!( plt; ylims=(0,0.4), yscale=:linear )

plot!( plt; xlabel="density, "*L"s", ylabel="noise amplitude, "*L"η" )

In [ ]:
# Import distribution data.
Fdatalist = [round.( defInt, vec( readdlm( folderdata[i,j]*"cluster-frequency.txt" ) ) )
    for i ∈ 1:Ns, j ∈ 1:Nη]
Pdatalist = Fdatalist./sum.( Fdatalist );

In [ ]:
# Extract the optimal power law of the distribution.
λdata = hcat( [
    ThreadsX.map( i -> 10.0.^optimize( createzipfloss( N, Fdatalist[i,j] ), [log10( 1.5 )] ).minimizer[1], 1:Ns )
    for j ∈ 1:Nη]... );

In [ ]:
# Compute the optimal distribution and JSD.
P̂datalist = hcat( ThreadsX.map(
    j -> [zipf.( 1:N; b=N, λ=λdata[i,j] ) for i ∈ 1:Ns], 1:Nη )... )
σdata = [jsd( Pdatalist[i,j], P̂datalist[i,j] )
    for i ∈ 1:Ns, j ∈ 1:Nη];

In [ ]:
# Weighting using the number of bins.
W0data = abundance.( Pdatalist; a=0 )./N   # Percentage occupied.
W1data = 1.0#evenness.( P̂datalist; a=2, b=1 )  # Hill's ratio.
Σdata = 1.0 .- σdata./(maximum( σdata ) - minimum( σdata ))

# Compute the order parameter and renormalize.
Hdata = W0data.*W1data.*Σdata
println( "'Best' power-law:  ", argmax( (W0data.*W1data.*Σdata) ).I )
println( "'Worst' power-law: ", argmin( (W0data.*W1data.*Σdata) ).I )

In [ ]:
# Plot variable.
plt = plot(; layout=@layout( [a{0.90w} b] ), size=(525,175), legend=false )
plot!( plt; left_margin=10pt, bottom_margin=10pt )
plot!( plt[2]; left_margin=0pt, right_margin=-5pt )

# Plot the order parameter for changing density.
plot!( plt, [0,ηlist[Nη]], [1,1]./2; color=:gray68, lw=2 )
for i ∈ 1:Ns
    plot!( plt[1], ηlist, μφdata[i,:]; color=colormap2[(i-1)/Ns], marker=:circ, lw=2 )
end

plot!( plt[1]; xlims=(0,ηlist[end]) )
plot!( plt[1]; ylims=(0,1), yticks=[0,0.5,1.0] )
plot!( plt[1]; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )
plot!( plt[1]; title="(a)", titleposition=:left )
plot!( plt[1]; framestyle=:box )

# Plot the color bar.
xmap = [0]
ymap = 0:0.01:(1+10/Ns)

heatmap!( plt[2], xmap, ymap, reshape(ymap, :, 1);
    color=colormap2, clims=(0,1), colorbar=false )
annotate!( plt[2], 0.90, 0.5,
    text( "density, "*L"s", 10, "Computer Modern", :center; rotation=-90 ), ymirror=true )

plot!( plt[2]; ylims=(0,1), yticks=([0,1], [L"10^{-2}", L"10^0"]) )
plot!( plt[2]; xticks=[] )
plot!( plt[2]; framestyle=:box )

# saveplot( plt, figurefolder*"order-transition_N-$(N).pdf"; background=:white, dpi=600 );

In [ ]:
is = 14:21;  js = fill( Nη - 7, length( is ) )
jη =  7:15;  iη = fill( Ns - 3, length( jη ) )

# Initialize plot.
plt = plot(; layout=@layout( [[a{0.90h}; b] [c; d]] ), size=(525,350), legend=false )
plot!( plt[1]; bottom_margin=0pt )
plot!( plt[2]; top_margin=0pt, bottom_margin=5pt )

# Build heatmap of the optimal JSD.
cmax = 0.1ceil( maximum( Hdata )/0.1 )
heatmap!( plt[1], slist, ηlist, Hdata'; cmap=colormap1, clims=(0,cmax), colorbar=false )
plot!( plt[1], slist, ηclist; color=:gray70, lw=1, marker=:circ, label="" )

plot!( plt[1]; xlims=10.0.^(-2,0), xscale=:log10 )
plot!( plt[1]; ylims=(0.0,0.4), yscale=:linear )
plot!( plt[1]; xlabel="speed, "*L"s", ylabel="noise amplitude, "*L"η", framestyle=:box )
plot!( plt[1]; title="(a)", titleposition=:left )

# Draw example parameter boxes (varying density).
plot!( plt[1], [slist[is];slist[is][end:-1:1];slist[is[1]]],
    [ηlist[js] .+ δη/2;ηlist[js] .- δη/2;ηlist[js[1]] .+ δη/2]; color=:mediumpurple, lw=2 )
# annotate!( plt[1], slist[is[3]], ηlist[js[1]] + 2δη,
#     text( "(b)", "Computer Modern", 11, :yellowgreen, :center ) )

# Draw example parameter boxes (varying noise).
plot!( plt[1], [slist[iη].*(10.0^(δs/2));slist[iη].*(10.0^(-δs/2));slist[iη[1]].*(10.0^(δs/2))],
    [ηlist[jη];ηlist[jη][end:-1:1];ηlist[jη[1]]]; color=:indianred, lw=2 )
# annotate!( plt[1], slist[iη[1]]*10^(-2.5δs), ηlist[jη[end]] - 1δη,
#     text( "(c)", "Computer Modern", 11, :darkolivegreen, :center ) )

# Plot the color bar.
xmap = 0:0.01:1.0
ymap = [0,1]
heatmap!( plt[2], xmap, ymap, hcat( xmap, xmap )'; color=colormap1, colorbar=false )

plot!( plt[2]; ylims=(0,1.0) )
plot!( plt[2]; xlabel="power-law likeness, "*L"ς" )
plot!( plt[2]; yticks=[], framestyle=:box )

# Plot the cluster mass distribution for changing density.
plot!( plt[3]; right_margin=10pt )

ŝ = [13, 15, 20]
println( "(s,η) = ($(slist[ŝ]),$(ηlist[js[1]]))" )
for i ∈ ŝ
    H = Hdata[i,js[1]];  color = colormap1[H/maximum( Hdata )]
    scatter!( plt[3], Pdatalist[i,js[1]]; color=color, markerstrokewidth=0 )
end

xticks = round.( defInt, 2.0.^(0:2:10) )
plot!( plt[3]; xlims=(1,N), xticks=(xticks, string.( xticks )), xscale=:log10 )
plot!( plt[3]; ylims=10.0.^(-8,0), yticks=10.0.^(-8:4:0), yscale=:log10 )

plot!( plt[3]; bordercolor=:mediumpurple )
plot!( plt[3]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[3]; title="(b)", titleposition=:left )

# Plot the cluster mass distributoin for changing noise.
plot!( plt[4]; right_margin=10pt )

η̂ = [15, 11, 7]
println( "(s,η,ηc) = ($(slist[iη[1]]),$(ηlist[η̂]),$(ηclist[iη[1]]))" )
for j ∈ η̂
    H = Hdata[iη[1],j];  color = colormap1[H/maximum( Hdata )]
    scatter!( plt[4], Pdatalist[iη[1],j]; color=color, markerstrokewidth=0 )
end

xticks = round.( defInt, 2.0.^(0:2:10) )
plot!( plt[4]; xlims=(1,N), xticks=(xticks, string.( xticks )), xscale=:log10 )
plot!( plt[4]; ylims=10.0.^(-8,0), yticks=10.0.^(-8:4:0), yscale=:log10 )

plot!( plt[4]; bordercolor=:indianred )
plot!( plt[4]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[4]; title="(c)", titleposition=:left )

plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"power-law-likeness-speed_N-$(N)"; background=:white, dpi=600 );

In [ ]:
# Plot the two "order" parameters against one another.
plt = plot(; size=(300,300), legend=false )

scatter!( plt, vec( Σdata ), vec( W0data.*W1data ); color=:gray )

plot!( plt, 0.5*[1,1], [0,1]; color=:black, linestyle=:dot, lw=2 )
plot!( plt, [0,1], 0.5*[1,1]; color=:black, linestyle=:dot, lw=2 )

plot!( plt; xlims=(0,1.0) )
plot!( plt; ylims=(0,1.0) )
plot!( plt; xlabel="likeness", ylabel="diversity" )

In [ ]:
if false
    # Save the computed power-law exponent.
    for i ∈ 1:Ns, j ∈ 1:Nη
        writedlm( folderdata[i,j]*"power-law.txt", λdata[i,j] )
    end
end